# CipherMark -- Config 2/3 : entrainement 256 bits sur Colab

Ce notebook reproduit `configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml` :
resolution reelle (256), `nbits=256` (au lieu des 64 par defaut de DistSeal), corpus
elargi et representatif (COCO + Kodak + BSDS + scikit-image).

**A faire avant de lancer** : `Runtime > Change runtime type > GPU (T4 ou mieux)`.

**Reprise entre sessions** : le checkpoint est sauve sur Google Drive
(`output_dir` pointe vers `/content/drive/...`). Si la session Colab se coupe
(deconnexion ~12h en gratuit), remontez Drive et relancez simplement la cellule
"6. Entrainement" telle quelle -- `train.py` reprend automatiquement depuis
`checkpoint.pth` (train.py:440-450), sans rien a changer.

## 1. Monter Google Drive
Indispensable : sans ca, le checkpoint disparait a chaque deconnexion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/ciphermark/runs/colab_256bits"

## 2. Cloner (ou mettre a jour) le depot
Branche `ciphermark`, qui contient les correctifs nbits=256 (sync CUDA, VideoWam,
attenuation None) et les configs de ce notebook.

In [ ]:
import os

REPO_URL = "https://github.com/ngueagho/distseal-meta.git"
REPO_DIR = "code-memoire"

if not os.path.exists(REPO_DIR):
    !git clone -b ciphermark {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Installer les dependances

In [ ]:
!pip install -q -r requirements.txt

## 4. Construire le corpus (reseau datacenter Colab -- pas la connexion locale)
Le meme telechargement plante/rampe en local (COCO ~19 Go) ; il passe normalement
sans probleme depuis Colab. Ajuster `COCO_MAX` a la baisse si le telechargement
est trop lent sur la session du jour.

In [ ]:
COCO_MAX = 5000

!python -m scripts.ciphermark.build_corpus --out corpus-colab-raw \
    --coco unlabeled2017 --coco-max {COCO_MAX} --kodak --bsds

## 5. Split train / val (90 / 10)

In [ ]:
import os, shutil

src = "corpus-colab-raw"
files = sorted(os.listdir(src))
os.makedirs("corpus-colab/train", exist_ok=True)
os.makedirs("corpus-colab/val", exist_ok=True)

for i, f in enumerate(files):
    dst = "corpus-colab/val" if i < len(files) // 10 else "corpus-colab/train"
    shutil.copy(os.path.join(src, f), os.path.join(dst, f))

print("train:", len(os.listdir("corpus-colab/train")))
print("val:  ", len(os.listdir("corpus-colab/val")))

## 6a. Calibration (5 min) -- a lancer avant le run long
Regarder la valeur `s / it` dans les logs pour estimer le temps reel des 20 000
pas prevus par la config, et ajuster `epochs`/`iter_per_epoch` ci-dessous si besoin.

In [ ]:
!torchrun --nproc_per_node=1 --standalone train.py \
    --config configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml \
    --output_dir {OUTPUT_DIR}_calib \
    --epochs 2 --iter_per_epoch 20

## 6b. Le run pour de vrai
Reexecuter cette cellule telle quelle apres une deconnexion : reprise automatique
depuis `{OUTPUT_DIR}/checkpoint.pth`.

In [ ]:
!torchrun --nproc_per_node=1 --standalone train.py \
    --config configs/training/ciphermark/posthoc_pixel_256bits_colab.yaml \
    --output_dir {OUTPUT_DIR}

## 7. Une fois l'entrainement termine
Le checkpoint est dans `{OUTPUT_DIR}/checkpoint.pth`, sur Drive -- recuperable
depuis n'importe quelle session pour l'evaluation d'integration CipherMark
(`CipherMarkWam` charge via `distseal.utils.cfg.setup_model`).